# T37 - 同花顺Excel导入财务指标

## 2. 数据结构设计

本章详细说明项目中涉及的数据结构设计，包括数据库表结构、数据流转换和指标分类。

## 2.1 数据库表依赖

### 输入表（MySQL）

| 表名 | 描述 | 主要字段 |
|------|------|----------|
| basicinfo_credit | 信用债基础信息 | trade_code, ths_issuer_name_cn_bond |
| basicinfo_finance | 金融债基础信息 | trade_code, ths_issuer_name_cn_bond |
| basicinfo_abs | ABS基础信息 | trade_code, ths_sponsor_to_original_righter_bond |

In [ ]:
# 查看MySQL输入表结构
from config import get_mysql_bond_engine, get_postgres_engine
from sqlalchemy import text

mysql_engine = get_mysql_bond_engine()
pg_engine = get_postgres_engine()

# 查询发行人代码
sql = """
SELECT 
    max(trade_code) AS trade_code,
    ths_issuer_name_cn_bond
FROM (
    SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_credit
    UNION ALL
    SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_finance
    UNION ALL
    SELECT trade_code, ths_issuer_name_cn_bond FROM basicinfo_abs
) SQ
GROUP BY ths_issuer_name_cn_bond
LIMIT 5
"""

try:
    df_issuers = pd.read_sql(sql, pg_engine)
    print("发行人代码示例:")
    display(df_issuers)
except Exception as e:
    print(f"查询失败: {e}")

### 输出表（PostgreSQL）

| 表名 | 格式 | 描述 | 主要字段 |
|------|------|------|----------|
| financial_indicators_wide | 宽表 | 原始财务数据 | thscode, dt, 指标1, 指标2, ... |
| financial_indicators_long | 长表 | 转换后数据 | dt, trade_code, 指标, value |

## 2.2 宽表结构设计

### 宽表格式（原始数据）

| 字段名 | 类型 | 描述 |
|--------|------|------|
| thscode | VARCHAR | 债券代码/发行人代码 |
| dt | VARCHAR | 报告期（如20230630） |
| ths_total_assets_bond | FLOAT | 总资产 |
| ths_currency_fund_bond | FLOAT | 货币资金 |
| ... | FLOAT | 其他150+个指标 |

In [ ]:
# 宽表结构示例
wide_table_example = pd.DataFrame({
    'thscode': ['101660001.IB', '101660002.IB', '101660003.IB'],
    'dt': ['20230630', '20230630', '20230630'],
    'ths_total_assets_bond': [123456.78, 234567.89, 345678.90],
    'ths_currency_fund_bond': [10000.00, 20000.00, 30000.00],
    'ths_account_receivable_bond': [5000.00, 6000.00, 7000.00],
    'ths_inventory_bond': [3000.00, 4000.00, 5000.00],
    'ths_total_liab_bond': [80000.00, 150000.00, 200000.00],
    'ths_total_owner_equity_bond': [43456.78, 84567.89, 145678.90]
})

print("宽表格式示例:")
display(wide_table_example)

## 2.3 长表结构设计

### 长表格式（转换后）

| 字段名 | 类型 | 描述 |
|--------|------|------|
| dt | VARCHAR | 报告期 |
| trade_code | VARCHAR | 债券代码 |
| 指标 | VARCHAR | 指标名称 |
| value | FLOAT | 指标值 |

In [ ]:
# 长表结构示例（从宽表转换）
long_table_example = wide_table_example.melt(
    id_vars=['dt', 'thscode'],
    var_name='指标',
    value_name='value'
)
long_table_example.rename(columns={'thscode': 'trade_code'}, inplace=True)

print("长表格式示例:")
display(long_table_example.head(10))

## 2.4 财务指标分类

### 2.4.1 资产负债表指标

In [ ]:
# 资产负债表指标配置
balance_sheet_indicators = {
    "资产类": [
        ("ths_total_assets_bond", "总资产"),
        ("ths_currency_fund_detail_sum_bond", "货币资金合计"),
        ("ths_account_receivable_bond", "应收账款"),
        ("ths_inventory_bond", "存货"),
        ("ths_fixed_asset_bond", "固定资产"),
        ("ths_total_current_assets_bond", "流动资产合计"),
    ],
    "负债类": [
        ("ths_st_borrow_bond", "短期借款"),
        ("ths_lt_loan_bond", "长期借款"),
        ("ths_bond_payable_bond", "应付债券"),
        ("ths_total_current_liab_bond", "流动负债合计"),
        ("ths_total_liab_bond", "负债合计"),
    ],
    "权益类": [
        ("ths_total_owner_equity_bond", "所有者权益合计"),
        ("ths_actual_received_capital_bond", "实收资本"),
        ("ths_undstrbtd_profit_bond", "未分配利润"),
    ]
}

print("资产负债表指标分类:")
for category, indicators in balance_sheet_indicators.items():
    print(f"\n{category}:")
    for code, name in indicators:
        print(f"  - {code}: {name}")

### 2.4.2 利润表指标

In [ ]:
# 利润表指标配置
income_statement_indicators = [
    ("ths_ebit_bond", "息税前利润"),
    ("ths_ebitda_bond", "息税折旧摊销前利润"),
    ("ths_ebit_ttm_bond", "EBIT TTM"),
    ("ths_ebitda_ttm_bond", "EBITDA TTM"),
    ("ths_continue_operate_net_profit_ttm_bond", "持续经营净利润TTM"),
]

print("利润表指标:")
for code, name in income_statement_indicators:
    print(f"  - {code}: {name}")

### 2.4.3 现金流量表指标

In [ ]:
# 现金流量表指标配置
cash_flow_indicators = [
    ("ths_cash_received_of_borrowing_bond", "借款收到的现金"),
    ("ths_cash_received_from_bond_issue_bond", "发行债券收到的现金"),
    ("ths_invest_income_cash_received_bond", "投资收益收到的现金"),
    ("ths_cash_received_of_sales_service_bond", "销售收到的现金"),
    ("ths_cash_pay_for_debt_bond", "偿还债务支付的现金"),
]

print("现金流量表指标:")
for code, name in cash_flow_indicators:
    print(f"  - {code}: {name}")

### 2.4.4 财务比率指标

In [ ]:
# 财务比率指标配置
financial_ratio_indicators = [
    ("ths_annaul_net_asset_yield_bond", "年净资产收益率"),
    ("ths_gross_selling_rate_bond", "销售毛利率"),
    ("ths_roe_avg_by_ths_bond", "ROE"),
    ("ths_asset_liab_ratio_bond", "资产负债率"),
]

print("财务比率指标:")
for code, name in financial_ratio_indicators:
    print(f"  - {code}: {name}")

## 2.5 数据转换流程

### 宽表转长表的melt操作

In [ ]:
def wide_to_long(df_wide, id_vars=['dt', 'thscode'], 
                 var_name='指标', value_name='value'):
    """
    宽表转长表
    
    Parameters:
    -----------
    df_wide : DataFrame
        宽表格式数据
    id_vars : list
        保持不变的列
    var_name : str
        变量名列名
    value_name : str
        值列名
        
    Returns:
    --------
    DataFrame : 长表格式数据
    """
    df_long = df_wide.melt(
        id_vars=id_vars,
        var_name=var_name,
        value_name=value_name
    )
    
    # 重命名thscode为trade_code
    df_long.rename(columns={'thscode': 'trade_code'}, inplace=True)
    
    return df_long

# 测试转换函数
df_long = wide_to_long(wide_table_example)
print("转换后的长表:")
display(df_long)

## 2.6 风险阈值配置

用于财务指标的风险评估和预警。

In [ ]:
from config import RISK_THRESHOLDS

print("风险阈值配置:")
for key, value in RISK_THRESHOLDS.items():
    print(f"  {key}: {value}")

def check_debt_risk(debt_ratio):
    """
    检查负债比率风险等级
    
    Parameters:
    -----------
    debt_ratio : float
        负债比率
        
    Returns:
    --------
    str : 风险等级（正常/预警/危险）
    """
    if debt_ratio >= RISK_THRESHOLDS['debt_ratio_danger']:
        return "危险"
    elif debt_ratio >= RISK_THRESHOLDS['debt_ratio_warning']:
        return "预警"
    else:
        return "正常"

def check_current_risk(current_ratio):
    """
    检查流动比率风险等级
    
    Parameters:
    -----------
    current_ratio : float
        流动比率
        
    Returns:
    --------
    str : 风险等级（正常/预警/危险）
    """
    if current_ratio <= RISK_THRESHOLDS['current_ratio_danger']:
        return "危险"
    elif current_ratio <= RISK_THRESHOLDS['current_ratio_warning']:
        return "预警"
    else:
        return "正常"

# 测试风险评估
print("\n风险评估测试:")
test_ratios = [0.5, 0.7, 0.9]
for ratio in test_ratios:
    print(f"  负债比率 {ratio}: {check_debt_risk(ratio)}")